<a href="https://colab.research.google.com/github/keyvanmahmoudi/LLM_Engineering_Course/blob/main/sessions/Session09_evaluating_and_improving_rag.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

<div dir="rtl" style="font-family: 'Vazirmatn FD', Vazirmatn, Tahoma, sans-serif;">

# جلسهٔ ۹: ارزیابی و بهبود RAG

### اهداف این جلسه

- ارزیابی جداگانه دو بخش RAG: بازیابی و تولید پاسخ
- محاسبه Hit Rate@k و MRR برای بازیابی
- پیاده‌سازی BM25 برای فارسی و ترکیب آن با جستجوی معنایی با Reciprocal Rank Fusion
- بهبود ترتیب نتایج با Cross-Encoder Reranker
- سنجش درستی، وفاداری به منابع و امتناع درست با LLM-as-a-Judge و تحلیل خطا

> این نوت‌بوک به GPU نیاز دارد. در Colab از منوی `Runtime → Change runtime type` گزینهٔ **T4 GPU** را انتخاب کنید.

</div>

<div dir="rtl" style="font-family: 'Vazirmatn FD', Vazirmatn, Tahoma, sans-serif;">

## ۱. RAG دو جای خراب شدن دارد

وقتی سیستم RAG پاسخ غلط می‌دهد، دو علت کلی وجود دارد:

1. **خطای بازیابی:** تکه حاوی جواب اصلاً به مدل نرسیده است.
2. **خطای تولید:** تکه درست رسیده، ولی مدل آن را بد خوانده، چیزی از خودش اضافه کرده یا به سؤال جواب نداده است.

راه‌حل این دو کاملاً متفاوت است، پس باید **جداگانه** اندازه‌گیری شوند. اگر بازیابی خراب است، عوض کردن مدل زبانی کمکی نمی‌کند.

| بخش | پرسش | معیار |
|---|---|---|
| بازیابی | آیا تکه درست در k نتیجه اول هست؟ | Hit Rate@k، MRR |
| تولید | آیا پاسخ درست است؟ | درستی (Correctness) در برابر پاسخ مرجع |
| تولید | آیا پاسخ فقط بر منابع تکیه دارد؟ | وفاداری (Faithfulness) |
| تولید | وقتی جواب نیست، امتناع می‌کند؟ | نرخ امتناع درست |

</div>

<div dir="rtl" style="font-family: 'Vazirmatn FD', Vazirmatn, Tahoma, sans-serif;">

## راه‌اندازی مدل زبانی

همه نوت‌بوک‌های این دوره از طریق API سازگار با OpenAI با مدل حرف می‌زنند. تابع `setup_llm` از فایل `llm_utils.py` دو حالت دارد:

- `setup_llm("ollama")`: مدل متن‌باز `qwen2.5:7b` را با Ollama روی همین ماشین اجرا می‌کند. اجرای اول چند دقیقه برای نصب و دانلود مدل طول می‌کشد. در Colab حتماً **T4 GPU** را فعال کنید.
- `setup_llm("api")`: به هر سرویس سازگار با OpenAI وصل می‌شود. نشانی، کلید و نام مدل را در بخش **Secrets** کولب (آیکون کلید در نوار کناری) با نام‌های `LLM_BASE_URL`، `LLM_API_KEY` و `LLM_MODEL` ذخیره کنید.

جزئیات کار این تابع در جلسه ۱ توضیح داده شده است.

</div>

In [ ]:
!pip -q install openai sentence-transformers rank_bm25 hazm
!wget -q -N https://raw.githubusercontent.com/keyvanmahmoudi/LLM_Engineering_Course/main/llm_utils.py

from llm_utils import setup_llm, chat

client, MODEL = setup_llm("ollama")      # or: setup_llm("api")

In [ ]:
import json

import numpy as np
import pandas as pd
from hazm import Normalizer, word_tokenize
from llm_utils import download_data
from sentence_transformers import CrossEncoder, SentenceTransformer

with open(download_data("novinkala_chunks.jsonl"), encoding="utf-8") as f:
    chunks = [json.loads(line) for line in f]
with open(download_data("rag_eval.jsonl"), encoding="utf-8") as f:
    eval_set = [json.loads(line) for line in f]

_normalizer = Normalizer(persian_numbers=False)
_DIGITS = str.maketrans("۰۱۲۳۴۵۶۷۸۹٠١٢٣٤٥٦٧٨٩", "01234567890123456789")


def normalize(text):
    return _normalizer.normalize(text).translate(_DIGITS)


print(len(chunks), "chunks,", len(eval_set), "evaluation questions")
print(eval_set[0])

<div dir="rtl" style="font-family: 'Vazirmatn FD', Vazirmatn, Tahoma, sans-serif;">

## ۲. معیارهای بازیابی

برای هر پرسش، فهرست رتبه‌بندی‌شده تکه‌ها را داریم. در جلسه ۷ تکه‌ای را مرتبط می‌دانستیم که از سند منبع پرسش باشد. این معیار آسانی است، چون هر سند چند تکه دارد و موضوع سندها از هم جداست. اینجا سخت‌گیرتریم: هر پرسش در فایل ارزیابی یک **عبارت شاهد (Evidence)** دارد، یعنی جمله‌ای از سند که پاسخ در آن است. فقط تکه‌ای مرتبط است که این عبارت را داشته باشد. در مجموعه ما برای هر پرسش دقیقاً یک تکه مرتبط وجود دارد.

- **Hit Rate@k:** در چند درصد پرسش‌ها دست‌کم یک تکه مرتبط در k نتیجه اول هست؟
- **MRR (Mean Reciprocal Rank):** میانگین معکوس رتبه اولین نتیجه مرتبط. اگر اولین نتیجه مرتبط در رتبه ۱ باشد امتیاز ۱، در رتبه ۲ امتیاز ۰٫۵ و در رتبه ۳ امتیاز ۰٫۳۳ است.

$$\text{MRR} = \frac{1}{|Q|} \sum_{q \in Q} \frac{1}{\text{rank}_q}$$

MRR علاوه بر پیدا شدن تکه درست، به **جایگاه** آن هم حساس است. این مهم است، چون تکه‌های بالاتر اثر بیشتری بر پاسخ دارند.

</div>

In [ ]:
def evaluate_retrieval(search_fn, ks=(1, 3, 5)):
    """search_fn(query) -> list of chunk indices, best first."""
    hits = {k: [] for k in ks}
    reciprocal_ranks = []
    for item in eval_set:
        ranking = search_fn(item["question"])
        evidence = normalize(item["evidence"])
        relevant = [i for i, idx in enumerate(ranking) if evidence in chunks[idx]["text"]]
        first = relevant[0] + 1 if relevant else None
        reciprocal_ranks.append(1 / first if first else 0)
        for k in ks:
            hits[k].append(first is not None and first <= k)
    return {**{f"hit@{k}": np.mean(v) for k, v in hits.items()}, "MRR": np.mean(reciprocal_ranks)}

<div dir="rtl" style="font-family: 'Vazirmatn FD', Vazirmatn, Tahoma, sans-serif;">

## ۳. بازیابی چگال (Dense)

همان بازیاب جلسه قبل: امبدینگ E5 و شباهت کسینوسی.

</div>

In [ ]:
embedder = SentenceTransformer("intfloat/multilingual-e5-base")
chunk_vectors = embedder.encode([f"passage: {c['text']}" for c in chunks], normalize_embeddings=True)


def dense_search(query, n=20):
    q = embedder.encode([f"query: {normalize(query)}"], normalize_embeddings=True)[0]
    return list(np.argsort(-(chunk_vectors @ q))[:n])


results = {"dense (E5)": evaluate_retrieval(dense_search)}
pd.DataFrame(results).T.round(3)

<div dir="rtl" style="font-family: 'Vazirmatn FD', Vazirmatn, Tahoma, sans-serif;">

## ۴. BM25: جستجوی کلیدواژه‌ای حرفه‌ای

**BM25** استاندارد جستجوی کلیدواژه‌ای است و موتورهایی مثل Elasticsearch از آن استفاده می‌کنند. نسبت به شمارش ساده کلمات مشترک دو بهبود دارد:

- **IDF:** کلمه‌های نادر مثل «اقساطی» وزن بیشتری از کلمه‌های رایج مثل «است» دارند.
- **اشباع و نرمال‌سازی طول:** تکرار زیاد یک کلمه امتیاز را بی‌نهایت بالا نمی‌برد و سندهای بلند امتیاز ناعادلانه نمی‌گیرند.

کیفیت BM25 برای فارسی به **توکن‌سازی** وابسته است. از توکن‌ساز hazm روی متن نرمال‌شده استفاده می‌کنیم و چند کلمه بسیار رایج (**Stopwords**) را حذف می‌کنیم.

</div>

In [ ]:
from rank_bm25 import BM25Okapi

STOPWORDS = set("و در به از که را با این آن است برای یا تا هم یک بر اگر هر شود می‌شود شده".split())


def tokenize(text):
    return [t for t in word_tokenize(normalize(text)) if t not in STOPWORDS and len(t) > 1]


bm25 = BM25Okapi([tokenize(c["text"]) for c in chunks])


def bm25_search(query, n=20):
    return list(np.argsort(-bm25.get_scores(tokenize(query)))[:n])


print(tokenize("رمز یک‌بار مصرف چند دقیقه اعتبار دارد؟"))
results["BM25"] = evaluate_retrieval(bm25_search)
pd.DataFrame(results).T.round(3)

<div dir="rtl" style="font-family: 'Vazirmatn FD', Vazirmatn, Tahoma, sans-serif;">

## ۵. جستجوی ترکیبی با Reciprocal Rank Fusion

جستجوی معنایی و BM25 نقاط قوت متفاوتی دارند: یکی معنا و مترادف را می‌فهمد و دیگری کلمه‌ها و عددهای دقیق را. امتیازهای خام آن‌ها قابل مقایسه نیستند، چون یکی شباهت کسینوسی بین ۰ و ۱ است و دیگری عددی بی‌حد. پس به جای امتیاز، **رتبه‌ها** را ترکیب می‌کنیم:

$$\text{RRF}(d) = \sum_{r \in \text{rankers}} \frac{1}{c + \text{rank}_r(d)}$$

ثابت $c$ معمولاً ۶۰ است. سندی که در هر دو فهرست رتبه خوبی دارد، امتیاز بالایی می‌گیرد. RRF ساده و بدون پارامتر یادگرفتنی است و در عمل بسیار خوب کار می‌کند.

</div>

In [ ]:
def rrf(rankings, c=60):
    scores = {}
    for ranking in rankings:
        for rank, idx in enumerate(ranking, 1):
            scores[idx] = scores.get(idx, 0) + 1 / (c + rank)
    return sorted(scores, key=scores.get, reverse=True)


def hybrid_search(query, n=20):
    return rrf([dense_search(query, n), bm25_search(query, n)])[:n]


results["hybrid (RRF)"] = evaluate_retrieval(hybrid_search)
pd.DataFrame(results).T.round(3)

<div dir="rtl" style="font-family: 'Vazirmatn FD', Vazirmatn, Tahoma, sans-serif;">

## ۶. Reranker: نگاه دقیق‌تر به نامزدها

مدل امبدینگ یک **Bi-Encoder** است: پرسش و تکه **جداگانه** به بردار تبدیل می‌شوند و فقط بردارها مقایسه می‌شوند. این سریع است، چون بردار تکه‌ها یک بار ساخته می‌شود، ولی مدل هیچ‌وقت پرسش و تکه را **کنار هم** نمی‌بیند.

**Cross-Encoder** پرسش و تکه را با هم به یک ترنسفورمر می‌دهد و مستقیماً امتیاز مرتبط بودن را پیش‌بینی می‌کند. این روش بسیار دقیق‌تر ولی کندتر است، چون برای هر جفت پرسش و تکه یک اجرای کامل مدل لازم است. پس معماری دومرحله‌ای رایج این است:

1. **بازیابی (Recall):** با روش سریع، ۲۰ تا ۵۰ نامزد پیدا کن.
2. **رتبه‌بندی دوباره (Precision):** نامزدها را با Cross-Encoder مرتب کن و چند تای اول را نگه دار.

از مدل چندزبانه `BAAI/bge-reranker-v2-m3` استفاده می‌کنیم.

</div>

In [ ]:
reranker = CrossEncoder("BAAI/bge-reranker-v2-m3", max_length=512)


def rerank_search(query, n=20, candidates_fn=hybrid_search):
    candidates = candidates_fn(query, n)
    scores = reranker.predict([(normalize(query), chunks[i]["text"]) for i in candidates])
    return [candidates[i] for i in np.argsort(-scores)]


results["hybrid + reranker"] = evaluate_retrieval(rerank_search)
retrieval_table = pd.DataFrame(results).T.round(3)
retrieval_table

In [ ]:
import matplotlib.pyplot as plt

ax = retrieval_table[["hit@1", "hit@3", "MRR"]].plot.barh(figsize=(8, 4), xlim=(0, 1))
ax.set_title("Retrieval quality on 30 Novin Kala questions")
plt.tight_layout()
plt.show()

<div dir="rtl" style="font-family: 'Vazirmatn FD', Vazirmatn, Tahoma, sans-serif;">

در تفسیر نتایج، درس جلسه ۵ را به یاد داشته باشید: با ۳۰ پرسش، هر پرسش حدود ۳ درصد است. معمولاً بیشترین بهبود Reranker در **Hit@1 و MRR** دیده می‌شود، یعنی تکه درست به رتبه اول می‌آید، نه در Hit@5.

## ۷. ارزیابی تولید پاسخ

حالا کل سیستم RAG را با بهترین بازیاب اجرا می‌کنیم و پاسخ‌ها را با دو داور می‌سنجیم:

- **درستی (Correctness):** مقایسه با پاسخ مرجع، با نمره ۱ تا ۵، مثل جلسه ۵.
- **وفاداری (Faithfulness):** آیا همه ادعاهای پاسخ در منابع داده‌شده وجود دارد؟ پاسخی ممکن است درست باشد ولی وفادار نباشد، یعنی از دانش قبلی مدل آمده باشد، یا برعکس.

</div>

In [ ]:
RAG_SYSTEM = """You are the customer-support assistant of Novin Kala, an Iranian online store.
Answer the customer's question in Persian, using ONLY the numbered sources provided.
Rules:
- Every factual sentence must end with the number of its source in brackets, e.g. [2].
- If the sources do not contain the answer, reply exactly: «متأسفانه در اسناد نوین‌کالا پاسخی برای این پرسش پیدا نکردم.»
- Do not use outside knowledge and do not guess numbers, prices or dates.
- Mention conditions and exceptions that apply. Be concise and polite."""


def rag_answer(question, k=4, search_fn=rerank_search):
    top = search_fn(question)[:k]
    sources = "\n\n".join(f"<source id=\"{n}\">\n{chunks[i]['text']}\n</source>"
                           for n, i in enumerate(top, 1))
    answer = chat(client, MODEL, f"{sources}\n\nCustomer question: {question}",
                  system=RAG_SYSTEM, temperature=0)
    return answer, sources


outputs = []
for item in eval_set:
    answer, sources = rag_answer(item["question"])
    outputs.append({**item, "prediction": answer, "sources": sources})
print(outputs[0]["question"], "\n->", outputs[0]["prediction"])

In [ ]:
from typing import Literal

from pydantic import BaseModel


class Correctness(BaseModel):
    reasoning: str
    score: Literal[1, 2, 3, 4, 5]


class Faithfulness(BaseModel):
    unsupported_claims: list[str]
    faithful: bool


CORRECTNESS_RUBRIC = """Grade a Persian customer-support answer against the REFERENCE answer.
5 = all facts of the reference are present and correct, nothing wrong is added.
4 = correct, a minor detail or condition is missing.
3 = partly correct; an important detail is missing or slightly wrong.
2 = mostly wrong or misleading.
1 = wrong, irrelevant, or a refusal although the reference has an answer.
Write a short reasoning first, then the score."""

FAITHFULNESS_RUBRIC = """You check whether an ANSWER is supported by the SOURCES.
List every factual claim in the answer that is NOT stated in the sources (ignore greetings
and the refusal sentence). Then set faithful=true only if the list is empty."""


def judge(rubric, schema, prompt):
    response = client.chat.completions.create(
        model=MODEL, temperature=0,
        messages=[{"role": "system", "content": rubric}, {"role": "user", "content": prompt}],
        response_format={"type": "json_schema",
                         "json_schema": {"name": schema.__name__, "schema": schema.model_json_schema()}})
    return schema.model_validate_json(response.choices[0].message.content)


for row in outputs:
    c = judge(CORRECTNESS_RUBRIC, Correctness,
              f"QUESTION: {row['question']}\nREFERENCE: {row['answer']}\nANSWER: {row['prediction']}")
    f = judge(FAITHFULNESS_RUBRIC, Faithfulness,
              f"SOURCES:\n{row['sources']}\n\nANSWER: {row['prediction']}")
    row.update(correctness=c.score, faithful=f.faithful, unsupported=f.unsupported_claims)

generation = pd.DataFrame(outputs)
print(f"mean correctness: {generation.correctness.mean():.2f} / 5")
print(f"answers with score >= 4: {(generation.correctness >= 4).mean():.0%}")
print(f"faithful answers: {generation.faithful.mean():.0%}")

<div dir="rtl" style="font-family: 'Vazirmatn FD', Vazirmatn, Tahoma, sans-serif;">

### امتناع درست

سیستم باید وقتی جواب در اسناد نیست، امتناع کند. چند پرسش خارج از دامنه می‌پرسیم و بررسی می‌کنیم جمله امتناع در پاسخ هست یا نه:

</div>

In [ ]:
out_of_scope = [
    "قیمت دلار امروز چنده؟",
    "نوین‌کالا شعبه حضوری در شیراز دارد؟",
    "آیا نوین‌کالا لپ‌تاپ دست دوم هم خرید می‌کند؟",
    "ساعت کاری انبار مرکزی چیست؟",
    "برای خرید با ارز دیجیتال چه کنم؟",
]
REFUSAL = "پاسخی برای این پرسش پیدا نکردم"
refusals = [REFUSAL in rag_answer(q)[0] for q in out_of_scope]
print(f"correct refusals: {sum(refusals)}/{len(out_of_scope)}")

<div dir="rtl" style="font-family: 'Vazirmatn FD', Vazirmatn, Tahoma, sans-serif;">

## ۸. تحلیل خطا

هر پاسخ با نمره پایین را بر اساس علتش دسته‌بندی می‌کنیم. اگر تکه حاوی عبارت شاهد در منابع نبوده، خطا از **بازیابی** است. در غیر این صورت از **تولید** است.

</div>

In [ ]:
def retrieval_ok(row):
    evidence = normalize(row["evidence"])
    return any(evidence in chunks[i]["text"] for i in rerank_search(row["question"])[:4])


failures = generation.query("correctness <= 3").copy()
failures["cause"] = ["generation" if retrieval_ok(r) else "retrieval" for _, r in failures.iterrows()]
print(failures.cause.value_counts().to_dict())
pd.set_option("display.max_colwidth", 80)
failures[["question", "answer", "prediction", "correctness", "cause"]]

<div dir="rtl" style="font-family: 'Vazirmatn FD', Vazirmatn, Tahoma, sans-serif;">

| علت | نشانه | راه‌حل‌های ممکن |
|---|---|---|
| بازیابی | تکه درست در منابع نیست | جستجوی ترکیبی، Reranker، تقسیم بهتر، بازنویسی پرسش |
| تولید: نادیده گرفتن منبع | جواب در منابع هست ولی پاسخ غلط است | مدل قوی‌تر، پرامپت دقیق‌تر، k کمتر |
| تولید: افزودن از خود | پاسخ وفادار نیست | قاعده سخت‌گیرانه‌تر، دمای صفر، بررسی ارجاع |
| تولید: امتناع بی‌جا | جواب هست ولی امتناع کرده | بررسی قاعده امتناع و کیفیت بازیابی |

**ابزار آماده:** کتابخانه [Ragas](https://docs.ragas.io) همین معیارها، یعنی Faithfulness، Answer Correctness و Context Recall، را با پیاده‌سازی استاندارد ارائه می‌دهد و با API سازگار با OpenAI هم کار می‌کند. وقتی منطق این معیارها را فهمیدید، استفاده از آن ساده است.

## ۹. جمع‌بندی

- بازیابی و تولید را جداگانه ارزیابی کنید. راه‌حل خطای هر کدام متفاوت است.
- Hit Rate@k پیدا شدن را می‌سنجد و MRR جایگاه را هم در نظر می‌گیرد.
- BM25 با توکن‌سازی درست فارسی، مکمل جستجوی معنایی است و RRF آن دو را بدون تنظیم پارامتر ترکیب می‌کند.
- Cross-Encoder Reranker ترتیب نامزدها را دقیق‌تر می‌کند: بازیابی سریع با Recall بالا، سپس رتبه‌بندی دقیق.
- درستی، وفاداری و امتناع درست سه بعد کیفیت پاسخ‌اند. تحلیل خطا نشان می‌دهد کجا را باید بهتر کرد.

</div>

<div dir="rtl" style="font-family: 'Vazirmatn FD', Vazirmatn, Tahoma, sans-serif;">

## تمرین‌ها

هر تمرین را در سلول کد زیر آن حل کنید. پاسخ پیشنهادی در پوشهٔ `solutions` آمده است.

۱. فهرست Stopwordها را حذف کنید و BM25 را دوباره ارزیابی کنید. سپس به جای `word_tokenize` از `split()` ساده استفاده کنید. کدام تغییر اثر بیشتری دارد؟
۲. به جای RRF، میانگین وزن‌دار امتیازهای نرمال‌شده، یعنی Min-Max، را امتحان کنید. با وزن‌های ۰٫۳، ۰٫۵ و ۰٫۷ برای جستجوی معنایی، کدام بهتر است؟
۳. ده پرسش جدید و دشوارتر، مثلاً ترکیبی یا محاوره‌ای، به مجموعه ارزیابی اضافه کنید و تمام ارزیابی را تکرار کنید. رتبه روش‌ها تغییر می‌کند؟
۴. پنج پاسخ را خودتان از نظر وفاداری بررسی کنید و با نظر داور مقایسه کنید. داور کجا اشتباه کرده است؟

</div>

<div dir="rtl" style="font-family: 'Vazirmatn FD', Vazirmatn, Tahoma, sans-serif;">

**تمرین ۱.** فهرست Stopwordها را حذف کنید و BM25 را دوباره ارزیابی کنید. سپس به جای `word_tokenize` از `split()` ساده استفاده کنید. کدام تغییر اثر بیشتری دارد؟

</div>

In [ ]:
# پاسخ خود را اینجا بنویسید

<div dir="rtl" style="font-family: 'Vazirmatn FD', Vazirmatn, Tahoma, sans-serif;">

**تمرین ۲.** به جای RRF، میانگین وزن‌دار امتیازهای نرمال‌شده، یعنی Min-Max، را امتحان کنید. با وزن‌های ۰٫۳، ۰٫۵ و ۰٫۷ برای جستجوی معنایی، کدام بهتر است؟

</div>

In [ ]:
# پاسخ خود را اینجا بنویسید

<div dir="rtl" style="font-family: 'Vazirmatn FD', Vazirmatn, Tahoma, sans-serif;">

**تمرین ۳.** ده پرسش جدید و دشوارتر، مثلاً ترکیبی یا محاوره‌ای، به مجموعه ارزیابی اضافه کنید و تمام ارزیابی را تکرار کنید. رتبه روش‌ها تغییر می‌کند؟

</div>

In [ ]:
# پاسخ خود را اینجا بنویسید

<div dir="rtl" style="font-family: 'Vazirmatn FD', Vazirmatn, Tahoma, sans-serif;">

**تمرین ۴.** پنج پاسخ را خودتان از نظر وفاداری بررسی کنید و با نظر داور مقایسه کنید. داور کجا اشتباه کرده است؟

</div>

In [ ]:
# پاسخ خود را اینجا بنویسید